In [0]:
df = spark.read.format("csv") .option("header", "true") .option("inferSchema", "true") \
    .load("abfss://raw@store1171.dfs.core.windows.net/patients/messy_patients_200_rows.csv")

df.show()

+---------+--------+----+------+-------------+------+
|PatientID|    Name| Age|Gender|      Disease|  Cost|
+---------+--------+----+------+-------------+------+
|        1|    NULL|71.0|     F|       Asthma| 760.0|
|        2|   Chris|59.0|  NULL| Hypertension| 870.0|
|       21|    Noah|NULL|     F|       Asthma| 698.0|
|        4|     Bob| 0.0|  NULL|     Diabetes| 262.0|
|       44|  Noah  |NULL|  NULL| Hypertension|  NULL|
|        6|    NULL|NULL|  NULL|     Diabetes| -43.0|
|        7|    Emma| 0.0|     M|Heart Disease|  NULL|
|        8|    Emma|NULL|     F|       Asthma|  NULL|
|        9|   David|78.0|     F|Heart Disease| 800.0|
|       48|  Olivia|NULL|     M|Heart Disease| 669.0|
|       11|    Liam|71.0|     M|       Asthma|  NULL|
|       12|    NULL|NULL|     M|       Asthma|  NULL|
|       13|   David|10.0|     F|       Cancer|  NULL|
|       14|     Bob|NULL|  NULL| Hypertension| 729.0|
|       15|    NULL|79.0|     M|     Diabetes| 395.0|
|       16|  Olivia|77.0|  N

Removing Nulls and Empty Names


In [0]:
df_Clean = df.filter(df.Name.isNotNull())
df = df.filter(df.Name != "")

Removing Invalid Age and Negative Cost From the data

In [0]:
# Remove invalid age
df = df.filter(df.Age > 0)

# Remove negative cost
df = df.filter(df.Cost > 0)

Removing Duplicated From Data and Trim Spaces In Name Section

In [0]:
from pyspark.sql.functions import trim, upper
df = df.dropDuplicates(["PatientID"])
df = df.withColumn("Name", upper(trim(df.Name)))

In [0]:
from pyspark.sql.functions import col, trim, upper

df_clean = df \
    .filter(col("PatientID").isNotNull()) \
    .filter(col("Name").isNotNull()) \
    .filter(trim(col("Name")) != "") \
    .filter(col("Age").isNotNull()) \
    .filter(col("Age") > 0) \
    .filter(col("Cost").isNotNull()) \
    .filter(col("Cost") > 0) \
    .withColumn("Name", upper(trim(col("Name")))) \
    .fillna({"Disease": "Unknown"}) \
    .dropDuplicates(["PatientID"])

df_clean.show(20)
print("Raw count:", df.count())
print("Clean count:", df_clean.count())

+---------+------+----+------+-------------+-----+
|PatientID|  Name| Age|Gender|      Disease| Cost|
+---------+------+----+------+-------------+-----+
|      148|   BOB|13.0|     F|       Cancer|403.0|
|       53|SOPHIA|73.0|  NULL|      Unknown|779.0|
|       26| CHRIS|68.0|  NULL|     Diabetes|432.0|
|       44|OLIVIA|88.0|     M| Hypertension|385.0|
|      182|  EMMA|81.0|     M|       Asthma| 74.0|
|       16|OLIVIA|77.0|  NULL| Hypertension| 40.0|
|        3|  NOAH|52.0|  NULL| Hypertension| 22.0|
|       48|OLIVIA|38.0|  NULL|       Cancer| 58.0|
|       41|  EMMA|16.0|  NULL|       Cancer|159.0|
|      179|  JOHN| 6.0|  NULL| Hypertension|230.0|
|       37|  EMMA|47.0|  NULL| Hypertension|918.0|
|        9| DAVID|78.0|     F|Heart Disease|800.0|
|       72|SOPHIA|76.0|  NULL|       Cancer|563.0|
|      217| ALICE|24.0|  NULL|Heart Disease|306.0|
|       23| CHRIS|70.0|     M|Heart Disease|123.0|
|       49| ALICE|14.0|  NULL|       Asthma|256.0|
|       87| ALICE|57.0|     F| 

Filling Null values to Unknown

In [0]:
df_clean = df_clean.fillna({"Gender": "Unknown"})
df_clean.show(20)

+---------+------+----+-------+-------------+-----+
|PatientID|  Name| Age| Gender|      Disease| Cost|
+---------+------+----+-------+-------------+-----+
|      148|   BOB|13.0|      F|       Cancer|403.0|
|       53|SOPHIA|73.0|Unknown|      Unknown|779.0|
|       26| CHRIS|68.0|Unknown|     Diabetes|432.0|
|       44|OLIVIA|88.0|      M| Hypertension|385.0|
|      182|  EMMA|81.0|      M|       Asthma| 74.0|
|       16|OLIVIA|77.0|Unknown| Hypertension| 40.0|
|        3|  NOAH|52.0|Unknown| Hypertension| 22.0|
|       48|OLIVIA|38.0|Unknown|       Cancer| 58.0|
|       41|  EMMA|16.0|Unknown|       Cancer|159.0|
|      179|  JOHN| 6.0|Unknown| Hypertension|230.0|
|       37|  EMMA|47.0|Unknown| Hypertension|918.0|
|        9| DAVID|78.0|      F|Heart Disease|800.0|
|       72|SOPHIA|76.0|Unknown|       Cancer|563.0|
|      217| ALICE|24.0|Unknown|Heart Disease|306.0|
|       23| CHRIS|70.0|      M|Heart Disease|123.0|
|       49| ALICE|14.0|Unknown|       Asthma|256.0|
|       87| 

Verifiying If data still have some null value in gender

In [0]:
df_clean.filter(col("Gender").isNull()).count()

0

Cleaned Data is saved in Silver folder

In [0]:
df_clean.write.mode("overwrite").parquet("abfss://silver@store1171.dfs.core.windows.net/patients_clean/")

In [0]:
dbutils.fs.ls("abfss://silver@store1171.dfs.core.windows.net/patients_clean/")

[FileInfo(path='abfss://silver@store1171.dfs.core.windows.net/patients_clean/_SUCCESS', name='_SUCCESS', size=0, modificationTime=1777942419000),
 FileInfo(path='abfss://silver@store1171.dfs.core.windows.net/patients_clean/_committed_1083497154901125500', name='_committed_1083497154901125500', size=232, modificationTime=1777940632000),
 FileInfo(path='abfss://silver@store1171.dfs.core.windows.net/patients_clean/_committed_3099261116765036687', name='_committed_3099261116765036687', size=221, modificationTime=1777942418000),
 FileInfo(path='abfss://silver@store1171.dfs.core.windows.net/patients_clean/_committed_5998430546771719538', name='_committed_5998430546771719538', size=123, modificationTime=1777854396000),
 FileInfo(path='abfss://silver@store1171.dfs.core.windows.net/patients_clean/_committed_vacuum9081426270978574505', name='_committed_vacuum9081426270978574505', size=96, modificationTime=1777940632000),
 FileInfo(path='abfss://silver@store1171.dfs.core.windows.net/patients_clea

In [0]:
jdbc_url = "jdbc:sqlserver://project1-retail-sql-server.database.windows.net:1433;database=free-sql-db-2795630"

connection_properties = {
    "user": "Naveen",
    "password": "N@v@@n78",
    "driver": "com.microsoft.sqlserver.jdbc.SQLServerDriver"
}

df_clean.write.jdbc(
    url=jdbc_url,
    table="patients_clean",
    mode="overwrite",
    properties=connection_properties)


Checking Raw and Clean Count

In [0]:
df = spark.read.format("csv") .option("header", "true") .option("inferSchema", "true") \
    .load("abfss://raw@store1171.dfs.core.windows.net/patients/messy_patients_200_rows.csv")

df.count()

---------------------------------------------------------------------------
UnknownException                          Traceback (most recent call last)
File <command-6728222548405572>, line 4
      1 df = spark.read.format("csv") .option("header", "true") .option("inferSchema", "true") \
      2     .load("abfss://raw@store1171.dfs.core.windows.net/patients/messy_patients_200_rows.csv")
----> 4 df.count()

File /databricks/spark/python/pyspark/sql/connect/dataframe.py:318, in DataFrame.count(self)
    315 def count(self) -> int:
    316     table, _ = self.agg(
    317         F._invoke_function("count", F.lit(1))
--> 318     )._to_table()  # type: ignore[operator]
    319     return table[0][0].as_py()

File /databricks/spark/python/pyspark/sql/connect/dataframe.py:1923, in DataFrame._to_table(self)
   1921 def _to_table(self) -> Tuple["pa.Table", Optional[StructType]]:
   1922     query = self._plan.to_proto(self._session.client)
-> 1923     table, schema, self._execution_info = self

In [0]:

print("Raw count:", df.count())
print("Clean count:", df_clean.count())

Raw count: 26
Clean count: 26


Null Verification

In [0]:

from pyspark.sql.functions import col
from pyspark.sql import functions as F
df_clean.select([F.sum(col(c).isNull()).alias(c) for c in df_clean.columns])


Checking for Invalid Values

In [0]:
df_clean.filter(col("Age") <= 0).count()
df_clean.filter(col("Cost") <= 0).count()

0

ML For Risk Level

In [0]:
from pyspark.sql.functions import when

df_ml = df_clean.withColumn("RiskLevel",
    when((col("Age") > 60) | (col("Cost") > 700), "High")
    .when((col("Age") > 40), "Medium")
    .otherwise("Low"))

df_ml.show()

+---------+------+----+-------+-------------+-----+---------+
|PatientID|  Name| Age| Gender|      Disease| Cost|RiskLevel|
+---------+------+----+-------+-------------+-----+---------+
|      148|   BOB|13.0|      F|       Cancer|403.0|      Low|
|       53|SOPHIA|73.0|Unknown|      Unknown|779.0|     High|
|       26| CHRIS|68.0|Unknown|     Diabetes|432.0|     High|
|       44|OLIVIA|88.0|      M| Hypertension|385.0|     High|
|      182|  EMMA|81.0|      M|       Asthma| 74.0|     High|
|       16|OLIVIA|77.0|Unknown| Hypertension| 40.0|     High|
|        3|  NOAH|52.0|Unknown| Hypertension| 22.0|   Medium|
|       48|OLIVIA|38.0|Unknown|       Cancer| 58.0|      Low|
|       41|  EMMA|16.0|Unknown|       Cancer|159.0|      Low|
|      179|  JOHN| 6.0|Unknown| Hypertension|230.0|      Low|
|       37|  EMMA|47.0|Unknown| Hypertension|918.0|     High|
|        9| DAVID|78.0|      F|Heart Disease|800.0|     High|
|       72|SOPHIA|76.0|Unknown|       Cancer|563.0|     High|
|      2

In [0]:
df_ml.write.mode("overwrite").parquet("abfss://gold@store1171.dfs.core.windows.net/patients_risk/")